In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch
%env UNSLOTH_DISABLE_FAST_GENERATION = 1
max_seq_length = 256
dtype = None
load_in_4bit = False

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_NEW')

In [ ]:
from datasets import load_dataset
LOAD_SPECIFIC_FILE = True
dataset_name = "1024m/LID"
if LOAD_SPECIFIC_FILE:
    file_path = "Data_Hackathon/LID-1000.parquet"
    dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
else:
    dataset = load_dataset(dataset_name, token=hf_token)["train"]
print(f"dataset  : {dataset_name}")
print(f"samples  : {len(dataset)}")
print(f"columns  : {dataset.column_names}")
print(f"size     : {dataset.dataset_size / 1024**2:.3f} MB")

Data_Hackathon/LID-1000.parquet:   0%|          | 0.00/379M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

dataset  : 1024m/LID
samples  : 67000
columns  : ['lang', 'text', 'source', 'ISO-693-3']
size     : 676.859 MB


In [ ]:
NUM_LABELS = len(dataset.unique("ISO-693-3"))
print(NUM_LABELS)
print(dataset.unique("ISO-693-3"))

67
['gle', 'vie', 'por', 'hrv', 'cym', 'ara', 'xho', 'zho', 'nep', 'ces', 'nld', 'urd', 'tel', 'guj', 'lit', 'kor', 'tur', 'pol', 'mlt', 'fra', 'pan', 'msa', 'deu', 'ibo', 'mar', 'hau', 'swe', 'ron', 'rus', 'zul', 'tam', 'yor', 'ben', 'heb', 'est', 'dan', 'srp', 'cat', 'mya', 'lao', 'jpn', 'slv', 'nor', 'bul', 'slk', 'mlg', 'ind', 'fin', 'ell', 'glg', 'swh', 'hin', 'khm', 'jav', 'eus', 'tha', 'eng', 'fas', 'ukr', 'amh', 'wol', 'lav', 'ita', 'spa', 'hun', 'tgl', 'sna']


In [ ]:
labels = sorted(dataset.unique("ISO-693-3"))
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

In [ ]:
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
import torch.nn as nn
model, tokenizer = FastModel.from_pretrained(
    model_name = "intfloat/multilingual-e5-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    full_finetuning = True,
    load_in_4bit = load_in_4bit,
)
if hasattr(model.classifier, "out_proj"):
    model.classifier.out_proj = nn.Linear(model.classifier.out_proj.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
else:
    model.classifier = nn.Linear(model.classifier.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
model.config.num_labels = NUM_LABELS
model.config.id2label = id2label
model.config.label2id = label2id

==((====))==  Unsloth 2026.3.11: Fast Xlm_Roberta patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at intfloat/multilingual-e5-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
model = FastModel.get_peft_model(model, r = 8, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
                                 lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth",
                                 random_state = 1024, use_rslora = False, loftq_config = None, task_type = "SEQ_CLS",)

Unsloth: Full finetuning is enabled, so .get_peft_model has no effect


In [ ]:
from datasets import ClassLabel
if isinstance(dataset, dict):
    dataset = dataset["train"]
dataset = dataset.cast_column("ISO-693-3", ClassLabel(names=sorted(dataset.unique("ISO-693-3"))))
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column="ISO-693-3")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_seq_length)
train_dataset = dataset['train'].map(tokenize_function, batched=True, num_proc=16)
val_dataset = dataset["test"].map(tokenize_function, batched=True, num_proc=16)
print(len(train_dataset))
print(len(val_dataset))

Casting the dataset:   0%|          | 0/67000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/60300 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/6700 [00:00<?, ? examples/s]

60300
6700


In [ ]:
train_dataset = train_dataset.rename_column("ISO-693-3", "label")
val_dataset = val_dataset.rename_column("ISO-693-3", "label")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes = np.unique(labels), y = labels)

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
train_dataset = train_dataset.remove_columns(["text", "lang", "source"])
val_dataset = val_dataset.remove_columns(["text", "lang", "source"])
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [ ]:
import json
def process_benchmark(file_path):
    ds = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
    lang_col = next(c for c in ds.column_names if c.lower() == "iso-693-3")
    text_col = next(c for c in ds.column_names if c.lower() == "text")
    ds = ds.filter(lambda x: x[lang_col] in label2id)
    ds = ds.map(lambda x: {"labels": label2id[x[lang_col]]})
    if text_col != "text":
        ds = ds.rename_column(text_col, "text")
    ds = ds.map(tokenize_function, batched=True, num_proc=16)
    keep = [c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"] if c in ds.column_names]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    ds.set_format("torch")
    return ds
benchmark_files = {
    "CommonLID": "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet",
    "FLORES":    "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet",
    "SmolSent":  "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet",
    "UDHRLID":   "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet",
}
benchmark_datasets = {name: process_benchmark(path) for name, path in benchmark_files.items()}
for name, ds in benchmark_datasets.items():
    print(f"{name}: {len(ds)} samples")

Data_Benchmarks_Filtered/filtered_benchm(…):   0%|          | 0.00/74.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/336325 [00:00<?, ? examples/s]

Map:   0%|          | 0/268682 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/268682 [00:00<?, ? examples/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/12550 [00:00<?, ? examples/s]

Map:   0%|          | 0/3987 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/3987 [00:00<?, ? examples/s]

CommonLID: 268682 samples
FLORES: 58696 samples
SmolSent: 8630 samples
UDHRLID: 3987 samples


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import json
eval_context = {"dataset_name": "val", "step": 0}
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    per_label = {}
    for lid in np.unique(labels):
        mask = labels == lid
        per_label[id2label[lid]] = f"{accuracy_score(labels[mask], preds[mask]):.3f}"
    name = eval_context["dataset_name"]
    step = eval_context["step"]
    with open(f"{name}-{step}-SCORES.json", "w") as f:
        json.dump(per_label, f, indent=2)
    return {"macro_f1": macro_f1}

In [ ]:
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported
import torch
class LIDTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_context["step"] = self.state.global_step
        if eval_dataset is None and isinstance(self.eval_dataset, dict):
            all_metrics = {}
            for name, ds in self.eval_dataset.items():
                eval_context["dataset_name"] = name
                m = super().evaluate(eval_dataset=ds, ignore_keys=ignore_keys, metric_key_prefix=f"eval_{name}")
                all_metrics.update(m)
            return all_metrics
        eval_context["dataset_name"] = metric_key_prefix
        return super().evaluate(eval_dataset=eval_dataset, ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix)
eval_datasets = {"val": val_dataset, **benchmark_datasets}
trainer = LIDTrainer(
    model=model,
    processing_class=tokenizer,
    eval_dataset=eval_datasets,
    train_dataset=train_dataset,
    args=TrainingArguments(
        per_device_train_batch_size=1440,
        per_device_eval_batch_size=512,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        eval_strategy="steps",
        eval_steps=0.1,
        lr_scheduler_type="linear",
        seed=1024,
        output_dir="outputs",
        report_to="none",
    ),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60,300 | Num Epochs = 1 | Total steps = 42
O^O/ \_/ \    Batch size per device = 1,440 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1440 x 1 x 1) = 1,440
 "-____-"     Trainable parameters = 559,959,107 of 559,959,107 (100.00% trained)


Step,Training Loss,Validation Loss,Val Loss,Val Macro F1,Commonlid Loss,Commonlid Macro F1,Flores Loss,Flores Macro F1,Smolsent Loss,Smolsent Macro F1,Udhrlid Loss,Udhrlid Macro F1
5,3.027400,No log,2.403568,0.955523,2.954745,0.441412,2.793044,0.710241,2.448377,0.701032,2.764771,0.759784
10,1.047400,No log,0.554507,0.985288,1.146647,0.590555,0.868860,0.939027,0.877411,0.720438,0.916664,0.915949
15,0.264400,No log,0.117748,0.985866,0.427731,0.618198,0.186841,0.942322,0.310115,0.721179,0.294286,0.917821
20,0.101500,No log,0.055208,0.989527,0.389709,0.615988,0.078927,0.964644,0.191990,0.808179,0.208159,0.944210
25,0.069100,No log,0.040810,0.989340,0.366355,0.614937,0.052939,0.968188,0.155857,0.836306,0.191614,0.937540
30,0.060100,No log,0.037338,0.991553,0.345618,0.626626,0.043046,0.974202,0.143205,0.939398,0.183352,0.952423
35,0.047300,No log,0.035012,0.991719,0.332733,0.627933,0.039503,0.975054,0.128355,0.948954,0.184072,0.949502
40,0.042400,No log,0.034224,0.991865,0.333202,0.626938,0.038913,0.975489,0.126219,0.950537,0.183786,0.951247


In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
test_text = dataset["test"][0]["text"]
true_label = id2label[int(val_dataset[0]["labels"])]
result = classifier(test_text, truncation=True, max_length=256)
print(f"text       : {test_text[:100]}")
print(f"true label : {true_label}")
print(f"predicted  : {result[0]['label']} ({result[0]['score']:.4f})")

Device set to use cuda:0


text       : قطار توربینی یا توربوترن (به انگلیسی: turbotrain) به قطاری که با نیروی محرکه توربینی حرکت کند گفته م
true label : fas
predicted  : fas (0.9814)


In [ ]:
model.save_pretrained("baseline_lora_v2")  # Local saving
tokenizer.save_pretrained("baseline_lora_v2")

('baseline_lora_v2/tokenizer_config.json',
 'baseline_lora_v2/special_tokens_map.json',
 'baseline_lora_v2/sentencepiece.bpe.model',
 'baseline_lora_v2/added_tokens.json',
 'baseline_lora_v2/tokenizer.json')

In [ ]:
model.push_to_hub("1024m/LIDL1Bv3", token = hf_token) # Online saving
tokenizer.push_to_hub("1024m/LIDL1Bv3", token = hf_token) # Online saving

README.md:   0%|          | 0.00/559 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lmej67w/model.safetensors:   0%|          |  606kB / 1.12GB            

Saved model to https://huggingface.co/1024m/LIDL1Bv3


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mp6wx_qd4_/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            